# 17차시 실습 — 내 서비스를 '계측'하기 (트레이싱)

> **day1 → day2 → day3 연속 실습.** day2에서 키운 **내 MVP의 AI 기능**(에이전트/LLM 호출)을, 이제 **운영 가능한 상태**로 만듭니다.
> 운영(LLMOps)의 1번 토대는 **관찰성(observability)** — *보이지 않으면 고칠 수 없습니다.*
> LLM 호출에 **트레이싱(trace)** 을 붙여 **입력 / 출력 / 지연 / 토큰**을 기록하고, **비용·지연 요약**까지 만들어 봅니다.

## 🧪 사용법
- 본 실습은 **Codex CLI**로, 이 노트북은 같은 개념을 **OpenAI API로 즉시 실행**하는 동반 자료입니다.
- **Langfuse 키가 없어도 전부 실행됩니다.** 먼저 키 없이 도는 **로컬 트레이서**로 개념을 잡고, 같은 함수에 **Langfuse `@observe()`** 를 얹는 법을 봅니다.
- 각 단계에 **⌨️ 터미널 Codex** 명령(복붙용)을 함께 적어 두었습니다. 위에서부터 **순서대로** 실행하세요 (`Shift+Enter`).

In [10]:
# ✅ 환경 준비 — 맨 처음 한 번만 (day1 노트북과 동일)
%pip install -q openai
import os, json, getpass
from openai import OpenAI
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY (화면에 안 보임): ")
client = OpenAI(); MODEL = "gpt-4o-mini"
print("준비 완료 ·", MODEL)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
준비 완료 · gpt-4o-mini


In [11]:
%pip install -q langfuse


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 오늘의 연속 시나리오

day1: 정해진 동작을 하는 앱을 만들었습니다.
day2: 같은 앱을 **에이전트**(추론·도구·루프)로 키웠습니다.
day3: 그 **내 서비스의 AI 호출**을 프로덕션에서 **운영**합니다 — 무슨 일이 있었는지 **보이게** 만드는 것이 시작입니다.

> 공통 예시 = `my_service()` — **내 MVP의 AI 기능 한 조각**. 구조는 동일하니 **당신 팀 MVP의 실제 호출**로 바꾸면 그대로 적용됩니다(STEP 7).

## STEP 1 — 로컬 트레이서 만들기 (`@trace`)

관찰성의 핵심은 **"무슨 일이 있었는지 기록"** 하는 것. 거창한 도구 없이도 데코레이터 하나로
함수의 **입력 / 출력 / 지연(latency) / 토큰**을 모아 **트레이스 트리**로 볼 수 있습니다.
아래 `@trace`는 Langfuse `@observe()`의 **축소판**입니다 — 같은 아이디어를 키 없이 직접 구현합니다.

⌨️ **터미널 Codex:** `codex "함수의 입력/출력/지연/토큰을 기록하고 중첩 호출을 트리로 출력하는 @trace 데코레이터를 만들어줘"`

In [12]:
import time, functools

TRACES = []        # 수집된 모든 관찰(observation) 기록
_STACK = []        # 현재 부모-자식(중첩) 관계를 추적

def trace(name=None):
    """함수의 입력/출력/지연/토큰을 기록하는 미니 트레이서 (Langfuse @observe()의 축소판)."""
    def deco(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            obs = {"name": name or fn.__name__, "depth": len(_STACK),
                   "input": {"args": args, "kwargs": kwargs},
                   "output": None, "latency_s": None, "tokens": None}
            TRACES.append(obs); _STACK.append(obs)
            t0 = time.perf_counter()
            try:
                result = fn(*args, **kwargs)
                if isinstance(result, tuple) and len(result) == 2:   # (텍스트, 토큰수)
                    obs["output"], obs["tokens"] = result
                else:
                    obs["output"] = result
                return result
            finally:
                obs["latency_s"] = round(time.perf_counter() - t0, 3)
                _STACK.pop()
        return wrapper
    return deco

def print_trace_tree():
    """수집된 트레이스를 트리로 출력."""
    print("🌳 트레이스 트리")
    for o in TRACES:
        pad = "  " * o["depth"]
        out = o["output"] if isinstance(o["output"], str) else str(o["output"])
        out = (out[:50] + "…") if len(out) > 50 else out
        tok = f" · {o['tokens']}tok" if o["tokens"] else ""
        print(f"{pad}└─ {o['name']}  ⏱{o['latency_s']}s{tok}")
        print(f"{pad}     in : {o['input']['args']} {o['input']['kwargs'] or ''}")
        print(f"{pad}     out: {out}")

print("트레이서 준비 완료 · @trace, print_trace_tree()")

트레이서 준비 완료 · @trace, print_trace_tree()


## STEP 2 — 내 서비스(`my_service`)를 `@trace`로 감싸기

`my_service()`는 **내 MVP의 AI 기능 한 조각**입니다(day2 에이전트의 핵심 LLM 호출 자리).
여기에 `@trace`를 붙이는 순간, 호출할 때마다 **입력·출력·지연·토큰**이 자동으로 기록됩니다.

⌨️ **터미널 Codex:** `codex "내 서비스의 LLM 호출 my_service에 @trace를 붙이고 응답과 사용 토큰을 함께 반환하게 해줘"`

In [13]:
@trace(name="my_service")        # ← 내 MVP의 AI 기능을 계측한다
def my_service(question: str):
    """내 서비스의 AI 호출 한 조각. (답변, 총토큰)을 반환 → @trace가 입력/출력/지연/토큰 기록."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
    )
    return resp.choices[0].message.content, resp.usage.total_tokens

TRACES.clear()
answer, tokens = my_service("우리 서비스에 들어온 사용자 질문 1건을 처리해줘.")
print("답변:", answer, "\n")
print_trace_tree()

답변: 물론입니다! 사용자 질문을 알려주시면 그에 대한 답변을 제공해드리겠습니다. 질문 내용을 말씀해 주세요. 

🌳 트레이스 트리
└─ my_service  ⏱1.562s · 50tok
     in : ('우리 서비스에 들어온 사용자 질문 1건을 처리해줘.',) 
     out: 물론입니다! 사용자 질문을 알려주시면 그에 대한 답변을 제공해드리겠습니다. 질문 내용을 말…


## STEP 3 — Langfuse 등가 코드 (`@observe`) — **자체 호스팅(Self-host)**

방금 만든 로컬 `@trace`는 **Langfuse**가 하는 일의 축소판입니다.
Langfuse는 오픈소스 LLM 관찰성 플랫폼으로 **트레이싱 + 평가 + 프롬프트 관리 + 대시보드**를 제공하고
**Trace**(요청 1건) · **Observation**(내부 단계) · **Session**(대화 묶음) · **Score**(품질 점수) 개념으로 정리합니다.
함수에 **`@observe()` 한 줄**만 붙이면 입력·출력·지연이 자동 캡처되어 **대시보드**에 나타납니다.

> **이 캠프는 Langfuse를 '자체 호스팅(Self-host)'으로 씁니다.** 클라우드 가입 없이 **내 PC의 Docker**로 직접 띄워
> 데이터를 외부로 보내지 않고 학습합니다. (Docker는 **0차시 환경 구성**에서 설치)
>
> **① Langfuse 띄우기 (터미널에서 한 번):**
> ```bash
> git clone https://github.com/langfuse/langfuse.git
> cd langfuse
> docker compose up -d          # 잠시 후 http://localhost:3000 에서 열림
> ```
> **②** 브라우저로 `http://localhost:3000` 접속 → **로컬 계정 가입** → 프로젝트 생성 →
> **Settings ▸ API Keys**에서 **Public/Secret 키** 2개 복사 →
> **③** 아래 셀 실행 전 환경변수 3개를 설정합니다(아래 코드가 안내). `LANGFUSE_HOST=http://localhost:3000`.
>
> **이 셀은 키가 없어도 안전하게 실행됩니다.** `LANGFUSE_PUBLIC_KEY`가 설정돼 있을 때만 실제로 전송합니다.

⌨️ **터미널 Codex:** `codex "my_service에 langfuse @observe()를 적용하고, 자체 호스팅(localhost:3000) Langfuse로 LANGFUSE 키가 있을 때만 전송하도록 가드해줘"`

In [14]:
# Langfuse 키가 있을 때만 설치/전송 (키 없으면 안내만 출력하고 넘어감)

print("Langfuse를 '자체 호스팅(Self-host)'으로 띄웁니다 (0차시에서 Docker 설치):")
print("     1) git clone https://github.com/langfuse/langfuse.git && cd langfuse")
print("     2) docker compose up -d            # http://localhost:3000 에서 열림")
print("     3) localhost:3000 가입(로컬 계정) → 프로젝트 생성 → Settings ▸ API Keys에서 키 2개 복사")
print("     4) 환경변수 설정 후 이 셀 재실행:")
print("        LANGFUSE_PUBLIC_KEY · LANGFUSE_SECRET_KEY · LANGFUSE_HOST=http://localhost:3000")
print("   참고: from langfuse.openai import openai 로 바꾸면 토큰·비용까지 자동 계측됩니다.")

# 자체 호스팅(Self-host): Docker로 직접 띄운 주소(0차시에서 Docker 설치). 기본 http://localhost:3000
from langfuse import observe, get_client
from langfuse.openai import openai 
os.environ["LANGFUSE_HOST"] = getpass.getpass("LANGFUSE_HOST (화면에 안 보임): ")
os.environ["LANGFUSE_PUBLIC_KEY"] = getpass.getpass("LANGFUSE_PUBLIC_KEY (화면에 안 보임): ")
os.environ["LANGFUSE_SECRET_KEY"] = getpass.getpass("LANGFUSE_SECRET_KEY (화면에 안 보임): ")

os.environ["LANGFUSE_DEBUG"] = "False" # 제대로 동작하는지 체크하려면 true로 놓고 debug 모드 실행

@observe()                          # ← 이 한 줄로 입력·출력·지연을 자동 캡처
def my_service_lf(question: str):
    resp = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": question}],
    )
    return resp.choices[0].message.content

print("답변:", my_service_lf("우리 서비스에 들어온 사용자 질문 1건을 처리해줘."))
get_client().flush()                # 짧게 도는 스크립트/노트북은 flush 필수
print(f"✅ 자체 호스팅 Langfuse({os.environ['LANGFUSE_HOST']}) 대시보드의 Traces 탭에서 방금 호출을 확인하세요.")

Langfuse를 '자체 호스팅(Self-host)'으로 띄웁니다 (0차시에서 Docker 설치):
     1) git clone https://github.com/langfuse/langfuse.git && cd langfuse
     2) docker compose up -d            # http://localhost:3000 에서 열림
     3) localhost:3000 가입(로컬 계정) → 프로젝트 생성 → Settings ▸ API Keys에서 키 2개 복사
     4) 환경변수 설정 후 이 셀 재실행:
        LANGFUSE_PUBLIC_KEY · LANGFUSE_SECRET_KEY · LANGFUSE_HOST=http://localhost:3000
   참고: from langfuse.openai import openai 로 바꾸면 토큰·비용까지 자동 계측됩니다.
답변: 물론입니다! 사용자 질문을 말씀해 주시면, 최대한 도움이 되도록 답변드리겠습니다. 질문을 적어주시면 시작하겠습니다.
✅ 자체 호스팅 Langfuse(http://localhost:3000) 대시보드의 Traces 탭에서 방금 호출을 확인하세요.


## STEP 4 — Langfuse로 비용·지연 자동 계측 + 중첩 트레이스

이제 요약을 **손으로 계산하지 않습니다.** `from langfuse.openai import openai` 로 부르면
Langfuse가 **모델·토큰·비용·지연을 자동 계측**해 대시보드에 쌓아줍니다.
또 `@observe()` 함수 안에서 다른 `@observe()` 함수를 부르면 **부모-자식(중첩) 트레이스**가 자동으로 그려집니다 —
아래 `pipeline`은 ① 내 서비스 호출 → ② 한국어로 다듬기, 두 LLM 호출을 감싼 **부모 트레이스**입니다.

> 키가 없으면 OpenAI 호출은 되고 Langfuse 전송만 생략됩니다. **STEP 3에서 키를 설정**했다면
> 실행 후 **`localhost:3000` ▸ Traces**에서 pipeline(부모) 아래 자식 호출·토큰·비용을 그대로 볼 수 있습니다.

⌨️ **터미널 Codex:** `codex "langfuse.openai로 호출해 토큰·비용을 자동 계측하고, @observe로 두 단계를 감싼 중첩 트레이스를 남겨줘"`

In [15]:
# Langfuse가 토큰·비용·지연을 자동 계측 — 손으로 만들던 가격표/집계가 필요 없어짐
from langfuse import observe, get_client
from langfuse.openai import openai      # ← 이 wrapper로 부르면 자동 계측(모델·토큰·비용·지연)
langfuse = get_client()
_LF_ON = bool(os.environ.get("LANGFUSE_PUBLIC_KEY"))   # STEP 3에서 키를 넣었는가

@observe(name="polish")                  # 자식 2
def polish(text: str):
    resp = openai.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"다음을 자연스러운 한국어 한 문장으로 다듬어줘:\n{text}"}],
    )
    return resp.choices[0].message.content, resp.usage.total_tokens

@observe(name="my_service_obs")          # 자식 1
def my_service_obs(question: str):
    resp = openai.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": question}],
    )
    return resp.choices[0].message.content, resp.usage.total_tokens

@observe(name="pipeline")                # 부모: 두 자식을 감싼다 → 중첩 트레이스가 자동 생성
def pipeline(question: str):
    draft, t1 = my_service_obs(question)   # 자식 1
    final, t2 = polish(draft)              # 자식 2
    return final, t1 + t2

final, total = pipeline("우리 서비스를 한 문장으로 소개해줘.")
print("최종:", final)
print(f"이번 pipeline 총 토큰: {total}  (비용·지연은 Langfuse가 자동 계산)")
langfuse.flush()                         # 노트북/짧은 스크립트는 flush 필수
print("📊 비용·토큰·지연은 Langfuse가 자동 계산합니다 (손으로 PRICE 표를 만들 필요 없음).")
print("→ 대시보드 Traces에서 pipeline(부모) 아래 my_service_obs·polish(자식)와 토큰·비용을 확인하세요." if _LF_ON
      else "→ (Langfuse 키 미설정) OpenAI 호출만 실행됨. STEP 3에서 키를 넣으면 대시보드로 전송됩니다.")

최종: 저희 서비스는 고객의 필요에 맞춘 맞춤형 솔루션을 제공하여 효율성과 만족도를 높여주는 플랫폼입니다.
이번 pipeline 총 토큰: 131  (비용·지연은 Langfuse가 자동 계산)
📊 비용·토큰·지연은 Langfuse가 자동 계산합니다 (손으로 PRICE 표를 만들 필요 없음).
→ 대시보드 Traces에서 pipeline(부모) 아래 my_service_obs·polish(자식)와 토큰·비용을 확인하세요.


## STEP 5 — SLO를 Langfuse **Score**로 남기기

SLO 준수 여부를 노트북 변수로만 재지 않고 **Langfuse Score**로 트레이스에 붙입니다 —
그래야 대시보드에서 "지연 SLO 준수율"을 **추세**로 보고, 나중에 품질 평가(Score)·알림으로 이어집니다.
`@observe()` 함수 안에서 **`langfuse.score_current_trace(...)` 한 줄**이면 됩니다.

⌨️ **터미널 Codex:** `codex "요청 지연을 재서 SLO 준수 여부를 langfuse score로 트레이스에 남겨줘"`

In [16]:
import time
SLO_LATENCY_MS = 3000          # 우리 팀 내부 목표 — 서비스에 맞게 조정

def record_slo(latency_ms):
    """SLO 준수 여부(0/1)를 Langfuse Score로 이 트레이스에 부착.
    langfuse 버전에 따라 score API 이름이 달라, 있는 것을 골라 안전하게 호출."""
    ok = 1 if latency_ms <= SLO_LATENCY_MS else 0
    comment = f"{latency_ms:.0f}ms / 목표 {SLO_LATENCY_MS}ms"
    try:
        langfuse.score_current_trace(name="latency_slo_ok", value=ok, data_type="BOOLEAN", comment=comment)
    except (AttributeError, TypeError):
        try:
            langfuse.create_score(name="latency_slo_ok", value=ok,
                                  trace_id=langfuse.get_current_trace_id(),
                                  data_type="BOOLEAN", comment=comment)
        except Exception as e:
            print(f"  (이 langfuse 버전은 score API가 달라 점수 전송은 생략: {type(e).__name__})")
    return ok

@observe(name="served_request")
def served_request(question: str):
    t0 = time.perf_counter()
    resp = openai.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": question}],
    )
    latency_ms = (time.perf_counter() - t0) * 1000
    record_slo(latency_ms)     # ← @observe가 만든 이 트레이스에 SLO 점수를 붙인다
    return resp.choices[0].message.content, latency_ms

_, ms = served_request("우리 서비스 SLO를 확인하는 테스트 요청.")
print(f"지연: {ms:.0f}ms · SLO({SLO_LATENCY_MS}ms) {'준수 ✅' if ms <= SLO_LATENCY_MS else '위반 ⚠️'}")
langfuse.flush()
print("→ 대시보드 Scores에서 latency_slo_ok 점수를 확인 (SLO=내부 목표, SLA=고객 약속)." if _LF_ON
      else "→ (키 미설정) 점수 전송은 생략. 키를 넣으면 Langfuse Score로 쌓입니다.")

지연: 6618ms · SLO(3000ms) 위반 ⚠️
→ 대시보드 Scores에서 latency_slo_ok 점수를 확인 (SLO=내부 목표, SLA=고객 약속).


## STEP 6 — LLMOps 성숙도 셀프 체크 (Level 0 → 2)

강의의 성숙도 모델 — 우리 팀이 지금 어디쯤인지 체크리스트로 진단합니다.

In [17]:
checklist = {                      # 우리 팀 상태로 바꿔보세요
    "트레이싱(관찰성)":        True,    # 오늘 붙였다!
    "골든셋 평가":             False,   # 15차시에서 만들었다면 True
    "프롬프트 버전 관리":      False,   # 19차시
    "알림 규칙":               False,   # 18차시
    "레드팀/보안 점검":        False,   # 21차시
}
score = sum(checklist.values())
level = 0 if score <= 1 else (1 if score <= 3 else 2)
for k, v in checklist.items(): print(f"  {'✅' if v else '⬜'} {k}")
print(f"\n우리 팀 성숙도: Level {level} ({score}/5) — Day3가 끝나면 전부 ✅가 목표")

  ✅ 트레이싱(관찰성)
  ⬜ 골든셋 평가
  ⬜ 프롬프트 버전 관리
  ⬜ 알림 규칙
  ⬜ 레드팀/보안 점검

우리 팀 성숙도: Level 0 (1/5) — Day3가 끝나면 전부 ✅가 목표


## STEP 7 — ⭐ 내 MVP에 적용 (Langfuse `@observe()`)

위 구조를 **그대로** 팀 MVP에 옮깁니다. 아래 `my_service_mine()` 안을 **내 MVP의 실제 AI 호출**
(day2 에이전트의 핵심 호출)로 바꾸고 `from langfuse.openai import openai`로 부르면,
**토큰·비용·지연·중첩이 자동으로 Langfuse에 쌓입니다.** 이게 저녁 팀 프로젝트 운영 모니터링의 첫 데이터입니다.

⌨️ **터미널 Codex:** `codex "내 MVP의 AI 호출을 langfuse @observe로 계측하고 SLO 점수를 남겨줘"`

In [18]:
# 팀별 작성: 내 MVP의 'AI 기능'을 여기에 연결 (day2 에이전트의 핵심 호출).
@observe(name="my_service_mine")
def my_service_mine(user_input: str):
    # TODO: 아래를 우리 서비스의 실제 호출로 교체 (예: day2 run_agent(user_input))
    resp = openai.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": user_input}],
    )
    return resp.choices[0].message.content

MY_INPUT = "여기에 우리 서비스에 들어올 법한 실제 요청을 적어보세요"  # ⬜ 팀별로 교체
out = my_service_mine(MY_INPUT)
print("내 MVP 응답:", out)
langfuse.flush()
print("→ 이 트레이스가 우리 팀 운영 모니터링의 첫 데이터입니다. (저녁 팀 프로젝트의 첫 커밋)")
print("   대시보드 Traces에서 my_service_mine의 입력·출력·토큰·비용을 확인하세요." if _LF_ON
      else "   (키 미설정) STEP 3에서 Langfuse 키를 넣으면 대시보드로 전송됩니다.")

내 MVP 응답: 물론입니다! 다음은 서비스에 들어올 법한 실제 요청 사례입니다:

1. **고객 지원 요청**
   - "안녕하세요, 최근에 주문한 상품의 배송 상태가 궁금합니다. 확인 부탁드립니다."

2. **기술 지원 요청**
   - "저희 웹사이트에 로그인할 수 없습니다. 비밀번호 재설정을 도와주실 수 있나요?"

3. **서비스 기능 문의**
   - "우리 회사의 새 서비스를 시작하고 싶은데, 어떤 기능이 있는지 자세히 설명해 주실 수 있나요?"

4. **결제 문제**
   - "결제 과정에서 오류가 발생했습니다. 어떻게 해결할 수 있을까요?"

5. **피드백 요청**
   - "최근에 이용한 서비스에 대해 피드백을 남기고 싶습니다. 어떤 방법으로 제출할 수 있나요?"

6. **서비스 변경 요청**
   - "현재 이용 중인 요금제를 변경하고 싶습니다. 어떤 절차가 필요한가요?"

7. **계정 관리 요청**
   - "제 계정 정보를 업데이트하고 싶은데, 어떤 정보를 제공해야 하나요?"

이런 요청들은 고객들이 서비스와 관련하여 자주 문의할 법한 예시입니다.
→ 이 트레이스가 우리 팀 운영 모니터링의 첫 데이터입니다. (저녁 팀 프로젝트의 첫 커밋)
   대시보드 Traces에서 my_service_mine의 입력·출력·토큰·비용을 확인하세요.


## 정리

- 운영(LLMOps)의 1번 토대는 **관찰성** — 요청마다 **입력/출력/지연/토큰**을 **트레이스**로 남긴다.
- `@trace`(로컬)로 원리를 익히고(STEP 1–2), **Langfuse `@observe()` + `langfuse.openai`** 로 **토큰·비용·지연·중첩을 자동 계측**했다(STEP 4).
- SLO는 **Langfuse Score**로 남겨 대시보드에서 추세로 본다(STEP 5) → 평가(Score)·프롬프트 관리·알림으로 이어진다. *보이지 않으면 고칠 수 없다.*